In [1]:
# Author: Kaifeng ZHU.
# This file is for R-DA-MPC control simulation result generation.
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots
from cell_slb import create_cell_from_stored_json
import theoratical_limit_calculation as tlc
import plotly.express as px

from run_search_then_apply_rolling_v2 import run_rolling_5y

# Define a color for each SOC limit
color_seq = px.colors.qualitative.Set1

from plot_config import (
    apply_scientific_theme,
    ENERGY_COLORS,
    style_trace_line,
    finalize_figure,
)

# Apply ploting theme.
apply_scientific_theme()

# Data Import

In [2]:
# Import prediction results and actual measurements.
pred_df = pd.read_csv("./Prediction_Result/pred_result_df_2.csv")
pred_df['datetime_utc'] = pd.to_datetime(pred_df['datetime_utc'])
pred_df['Demand'] = pred_df['Demand']/1000
pred_df['Demand_Predicted'] = pred_df['Demand_Predicted']/1000

# DOUBLE the PV Power here !!!
pred_df['PV'] = pred_df['PV']/1000 
pred_df['PV_Predicted'] = pred_df['PV_Predicted']/1000 


# 1) 取出你要复制的历史时段：2020-12-27 ~ 2022-12-26
hist_block = pred_df.set_index('datetime_utc').loc['2020-12-27':'2022-12-26']

# 2) 平移 3 年，得到 2023-12-27 ~ 2025-12-26（此时没有 2024-02-29）
future_block = hist_block.copy()
future_block.index = future_block.index + pd.DateOffset(years=3)

# 3) 为 2024-02-29 补一整天数据：复制前一天 2024-02-28 的 24 个小时
day_28 = future_block.loc['2024-02-28'].copy()      # 2024-02-28 00:00~23:00
day_28.index = day_28.index + pd.Timedelta(days=1)  # 改成 2024-02-29 00:00~23:00

# 拼回未来两年的 block，并按时间排序
future_block = pd.concat([future_block, day_28]).sort_index()

# 4) 将未来两年拼接回原始 pred_df
pred_full = pd.concat(
    [pred_df.set_index('datetime_utc'), future_block],
    axis=0
)

# 如果时间轴上有重复（理论上没有），保留第一次出现的
pred_full = pred_full[~pred_full.index.duplicated(keep='first')]

# 恢复成有 datetime_utc 列的 DataFrame
pred_full = pred_full.reset_index().rename(columns={'index': 'datetime_utc'})

price_df = pd.read_csv('Processed_Dataset/price_df.csv')
price_df.rename(columns={'Unnamed: 0': 'datetime_utc'}, inplace=True)
price_df['datetime_utc'] = pd.to_datetime(price_df['datetime_utc'])

# 统一列名：时间列叫 datetime_utc，电价列叫 Price（方便后面 join）
price_df.columns = ['datetime_utc', 'Price']

price_df = price_df.sort_values('datetime_utc').set_index('datetime_utc')

# 与 pred_full 对齐时间范围
start_dt = pred_full['datetime_utc'].min()
end_dt   = pred_full['datetime_utc'].max()
price_df = price_df.loc[start_dt:end_dt]

# 用真实 Price 覆盖 pred_full 中的 Price
pred_full = pred_full.sort_values('datetime_utc').set_index('datetime_utc')

# 删掉原来的 Price（如果有的话）
pred_full = pred_full.drop(columns=['Price'], errors='ignore')

# 按时间对齐，把真实电价拼进来
pred_full = pred_full.join(price_df, how='left')

# 恢复成普通 DataFrame
pred_full = pred_full.reset_index()  # datetime_utc 变回列

test_df = pred_full.set_index('datetime_utc').loc['2020-12-27':'2025-12-26'].reset_index()
test_df

,datetime_utc,Demand,PV,Demand_Predicted,PV_Predicted,Price
0,2020-12-27 00:00:00,172.910835,0.005080,154.463330,0.0,-0.00687
1,2020-12-27 01:00:00,99.722889,0.003671,170.890810,0.0,-0.01245
2,2020-12-27 02:00:00,141.623719,0.001805,155.713920,0.0,-0.01338
3,2020-12-27 03:00:00,140.699362,0.004986,156.995160,0.0,-0.01909
4,2020-12-27 04:00:00,131.699440,0.003033,164.521890,0.0,-0.01760
...,...,...,...,...,...,...
43819,2025-12-26 19:00:00,-5.826884,0.001291,43.957945,0.0,0.11626
43820,2025-12-26 20:00:00,-15.169414,0.001820,33.617590,0.0,0.11373
43821,2025-12-26 21:00:00,-23.261297,0.001524,22.813555,0.0,0.11150
43822,2025-12-26 22:00:00,-42.056562,0.000738,14.908608,0.0,0.10826


In [3]:
# # Assign buy, sell price typically.
# pred_full["Buy_Price"] = pred_full["Price"]
# pred_full["Sell_Price"] = 0.9 * pred_full["Price"]

In [4]:
pred_full

,datetime_utc,Demand,PV,Demand_Predicted,PV_Predicted,Price
0,2020-08-19 00:00:00,278.909446,-0.000000,237.115530,0.000000,0.03753
1,2020-08-19 01:00:00,273.983406,-0.000000,285.131200,0.000000,0.03391
2,2020-08-19 02:00:00,295.339601,-0.000000,274.959440,0.000000,0.03252
3,2020-08-19 03:00:00,310.412627,-0.000000,285.395560,0.000000,0.03048
4,2020-08-19 04:00:00,302.836874,4.510900,317.464500,17.945465,0.03025
...,...,...,...,...,...,...
46939,2025-12-26 19:00:00,-5.826884,0.001291,43.957945,0.000000,0.11626
46940,2025-12-26 20:00:00,-15.169414,0.001820,33.617590,0.000000,0.11373
46941,2025-12-26 21:00:00,-23.261297,0.001524,22.813555,0.000000,0.11150
46942,2025-12-26 22:00:00,-42.056562,0.000738,14.908608,0.000000,0.10826


In [5]:
# Assign simulation dfs.
# For transformer predictions.
normal_df = pred_full.copy()

# For perfect prediction.
pred_perfect_df = pred_full.copy()
pred_perfect_df['Demand_Predicted'] = pred_perfect_df['Demand']
pred_perfect_df['PV_Predicted'] = pred_perfect_df['PV']

# For past day usage.
pred_full_sorted = pred_full.sort_values('datetime_utc').reset_index(drop=True)
pred_prev_day = pred_full_sorted.copy()
pred_prev_day['Demand_Predicted'] = pred_full_sorted['Demand'].shift(24)
pred_prev_day['PV_Predicted'] = pred_full_sorted['PV'].shift(24)
pred_prev_day_clean = pred_prev_day.dropna(subset=['Demand_Predicted', 'PV_Predicted']).reset_index(drop=True)

ready_to_test_dict = {
    "TP": normal_df,
    "PP": pred_perfect_df,
    "PDU": pred_prev_day_clean
}

# Assign a battery system and baseline conditions
# datetime_utc, Demand, PV, Price
df = test_df.copy()

dt_h = 1.0  # 小时分辨率

# 1) 无电池净负荷（kW）：正=需要从电网买；负=PV多余可上网
df["Net_Load"] = df["Demand"] - df["PV"]

# 2) 分解成买电与卖电功率（kW）
df["Grid_Import_kW"] = df["Net_Load"].clip(lower=0.0)
df["Grid_Export_kW"] = (-df["Net_Load"]).clip(lower=0.0)

# 3) 转成电量（kWh）
df["Grid_Import_kWh"] = df["Grid_Import_kW"] * dt_h
df["Grid_Export_kWh"] = df["Grid_Export_kW"] * dt_h

# 4) Baseline cost（buy=sell=Price）
# 等价写法：Cost = Price * Net_Load * dt
# - 若 Net_Load>0（买电），Cost=Price*买电量
# - 若 Net_Load<0（卖电），Cost 为负（代表收入）
df["Baseline_Cost"] = df["Price"] * df["Net_Load"] * dt_h

total_baseline_cost = df["Baseline_Cost"].sum()

# 看看结果
cols = ["datetime_utc","Demand","PV","Price","Net_Load",
        "Grid_Import_kW","Grid_Export_kW","Baseline_Cost"]
print(df[cols].head())
print("Total baseline cost =", total_baseline_cost)


# Build the battery system.
system_kwargs = dict(
    cache_dir = "./battery_model_params_20260214/1_hour_0.0_1.0",
    Ns=96,  # Number of series cells.
    Np=10,  # Number of parallel packs.
    N_series=105,  # Number of series cells in each pack.
    N_parallel=2,
)

system = tlc.build_system(**system_kwargs)

# ===========
# Calculate the installation cost of SLBs.
# ===========
system_nominal_capacity_Ah = system.nominal_capacity_Ah
# Get current energy capacity of the battery system.
sys_capacity_kwh = system_nominal_capacity_Ah * 3.6/1000 * 0.8
# slb_price = 106
slb_price = 79.2    # 70% off.
slb_installation_cost = slb_price * sys_capacity_kwh
target_percentage = slb_installation_cost / total_baseline_cost
print(f"Target saving percentage: {target_percentage:.2%}")

         datetime_utc      Demand        PV    Price    Net_Load  \
0 2020-12-27 00:00:00  172.910835  0.005080 -0.00687  172.905755   
1 2020-12-27 01:00:00   99.722889  0.003671 -0.01245   99.719218   
2 2020-12-27 02:00:00  141.623719  0.001805 -0.01338  141.621914   
3 2020-12-27 03:00:00  140.699362  0.004986 -0.01909  140.694376   
4 2020-12-27 04:00:00  131.699440  0.003033 -0.01760  131.696407   

   Grid_Import_kW  Grid_Export_kW  Baseline_Cost  
0      172.905755             0.0      -1.187863  
1       99.719218             0.0      -1.241504  
2      141.621914             0.0      -1.894901  
3      140.694376             0.0      -2.685856  
4      131.696407             0.0      -2.317857  
Total baseline cost = 783448.865516932
Target saving percentage: 12.33%


In [6]:
# Typical R-DA-MPC.
# Run the simulation.
# For benchmark comparison.
normal_df["Buy_Price"] = normal_df["Price"]
normal_df["Sell_Price"] = normal_df["Price"]
result_df, calib_log_df = run_rolling_5y(
    df_raw=normal_df,
    cache_dir="./battery_model_params_20260214/1_hour_0.0_1.0",
    Ns=96, Np=10, N_series=105, N_parallel=2,
    start="2020-12-27",
    end="2025-12-26",
    past_weeks=16,              
    soc_grid_size=101,
    action_grid_size=81,
    allow_export=True,
    terminal_equal_init=False,
    n_jobs=-1,
    max_iter_calib=10,
    max_iter_day=10,
    calib_every_days=1,
    use_calibrated_lambda_only=True,   # ← 新增：禁用每日 λ 求解，只用校准 λ，
    return_calibration_log=True,
)

hourly_df_all = result_df.copy()

# ----- 1) baseline_cost_cum：无电池时的累积电费 -----
dt_h = 1.0
if "baseline_cost_cum" not in hourly_df_all.columns:
    net_load = hourly_df_all["Demand"].to_numpy(float) - hourly_df_all["PV"].to_numpy(float)
    baseline_cost_hourly = hourly_df_all["Buy_Price"].to_numpy(float) * net_load * dt_h
    hourly_df_all["baseline_cost_cum"] = np.cumsum(baseline_cost_hourly)

hourly_df_all['grid_cost_cum'] = np.cumsum(hourly_df_all['grid_cost'])

# ----- 2) SOH：由 cell 的 EFC–SOH 关系根据 EFC_cum_cell 插值 -----
if "SOH" not in hourly_df_all.columns:
    try:
        cell = system.pack.cell
    except NameError:
        cache_dir = "./battery_model_params_20260214/1_hour_0.0_1.0"
        cell = create_cell_from_stored_json(cache_dir, show_info=False)

    deg = cell.degradation_df
    efc_index = deg.index.to_numpy(float)
    soh_values = deg.values if isinstance(deg, pd.Series) else deg.iloc[:, 0].to_numpy(float)

    efc_cum = hourly_df_all["efc_cycle"].to_numpy(float)
    hourly_df_all["SOH"] = np.interp(efc_cum, efc_index, soh_values)

# hourly_df_all.to_csv("./Prediction_Accuracy_Analysis/MF-1.csv", index=False)
print('R-DA-MPC#1 done.')
print(f'Performance: {1 - (hourly_df_all["grid_cost"].sum() / total_baseline_cost):.2%}')


R-DA-MPC#1 done.
Performance: 10.09%


In [7]:
# hourly_df_all.to_csv("./Operation_Comparison_Result/R-DA-MPC#1.csv", index=False)
# calib_log_df.to_csv("./Operation_Comparison_Result/R-DA-MPC_log/R-DA-MPC#1_log.csv", index=False)

In [8]:
hourly_df_all.columns

Index(['datetime_utc', 'Demand', 'PV', 'Buy_Price', 'Sell_Price',
       'Demand_Predicted', 'PV_Predicted', 'p_batt_request_kw',
       'p_batt_applied_kw', 'env_power_limit_kw', 'env_power_limit_charge_kw',
       'env_power_limit_discharge_kw', 'soc_power_limit_kw', 'soc',
       'p_grid_kw', 'grid_cost', 'grid_cost_cum', 'soh', 'efc_cycle',
       'lambda_star', 'lambda_calib', 'target_efc_window', 'efc_used',
       'efc_used_window_dp', 'baseline_cost_cum', 'SOH'],
      dtype='object')

## Market friction and prediction accuracy analysis model simulation

In [9]:
# Run the simulation.
# For the baseline analysis result.

market_friction_search_list = [0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6, 0.55, 0.5]

result_summary_df = pd.DataFrame(columns=['Model Name', 'Market Friction', 'Saving Percentage', 'EFC'])

normal_df["Buy_Price"] = normal_df["Price"]
normal_df["Sell_Price"] = normal_df["Price"]
result_df, calib_log_df = run_rolling_5y(
    df_raw=normal_df,
    cache_dir="./battery_model_params_20260214/1_hour_0.0_1.0",
    Ns=96, Np=10, N_series=105, N_parallel=2,
    start="2020-12-27",
    end="2025-12-26",
    past_weeks=4,              
    soc_grid_size=101,
    action_grid_size=81,
    allow_export=True,
    terminal_equal_init=False,
    n_jobs=-1,
    max_iter_calib=10,
    max_iter_day=10,
    calib_every_days=1,
    use_calibrated_lambda_only=True,   # ← 新增：禁用每日 λ 求解，只用校准 λ，
    return_calibration_log=True,
)

hourly_df_all = result_df.copy()

# ----- 1) baseline_cost_cum：无电池时的累积电费 -----
dt_h = 1.0
if "baseline_cost_cum" not in hourly_df_all.columns:
    net_load = hourly_df_all["Demand"].to_numpy(float) - hourly_df_all["PV"].to_numpy(float)
    baseline_cost_hourly = hourly_df_all["Buy_Price"].to_numpy(float) * net_load * dt_h
    hourly_df_all["baseline_cost_cum"] = np.cumsum(baseline_cost_hourly)

hourly_df_all['grid_cost_cum'] = np.cumsum(hourly_df_all['grid_cost'])

# ----- 2) SOH：由 cell 的 EFC–SOH 关系根据 EFC_cum_cell 插值 -----
if "SOH" not in hourly_df_all.columns:
    try:
        cell = system.pack.cell
    except NameError:
        cache_dir = "./battery_model_params_20260214/1_hour_0.0_1.0"
        cell = create_cell_from_stored_json(cache_dir, show_info=False)

    deg = cell.degradation_df
    efc_index = deg.index.to_numpy(float)
    soh_values = deg.values if isinstance(deg, pd.Series) else deg.iloc[:, 0].to_numpy(float)

    efc_cum = hourly_df_all["efc_cycle"].to_numpy(float)
    hourly_df_all["SOH"] = np.interp(efc_cum, efc_index, soh_values)

hourly_df_all.to_csv(f"./Prediction_Accuracy_Result/Simulation Result/MF-1.csv", index=False)
calib_log_df.to_csv(f"./Prediction_Accuracy_Result/Simulation Log/MF-1.csv", index=False)

print(f'Saved MF-1 Model')
print(f'Performance: {1 - (hourly_df_all["grid_cost"].sum() / total_baseline_cost):.2%}')
result_summary_df.loc[len(result_summary_df)] = [f'MF-1', 1, 1 - (hourly_df_all["grid_cost"].sum() / total_baseline_cost), hourly_df_all.iloc[len(hourly_df_all)-1]['efc_cycle']]

for market_friction in market_friction_search_list:
    for model_name, df_raw in ready_to_test_dict.items():
        df_raw["Buy_Price"] = df_raw["Price"]
        df_raw["Sell_Price"] = market_friction * df_raw["Price"]
        result_df, calib_log_df = run_rolling_5y(
            df_raw=df_raw,
            cache_dir="./battery_model_params_20260214/1_hour_0.0_1.0",
            Ns=96, Np=10, N_series=105, N_parallel=2,
            start="2020-12-27",
            end="2025-12-26",
            past_weeks=4,              
            soc_grid_size=101,
            action_grid_size=81,
            allow_export=True,
            terminal_equal_init=False,
            n_jobs=-1,
            max_iter_calib=10,
            max_iter_day=10,
            calib_every_days=1,
            use_calibrated_lambda_only=True,   # ← 新增：禁用每日 λ 求解，只用校准 λ，
            return_calibration_log=True,
        )

        hourly_df_all = result_df.copy()
        # ----- 1) baseline_cost_cum：无电池时的累积电费 -----
        dt_h = 1.0
        if "baseline_cost_cum" not in hourly_df_all.columns:
            net_load = hourly_df_all["Demand"].to_numpy(float) - hourly_df_all["PV"].to_numpy(float)
            baseline_cost_hourly = hourly_df_all["Buy_Price"].to_numpy(float) * net_load * dt_h
            hourly_df_all["baseline_cost_cum"] = np.cumsum(baseline_cost_hourly)

        hourly_df_all['grid_cost_cum'] = np.cumsum(hourly_df_all['grid_cost'])

        # ----- 2) SOH：由 cell 的 EFC–SOH 关系根据 EFC_cum_cell 插值 -----
        if "SOH" not in hourly_df_all.columns:
            try:
                cell = system.pack.cell
            except NameError:
                cache_dir = "./battery_model_params_20260214/1_hour_0.0_1.0"
                cell = create_cell_from_stored_json(cache_dir, show_info=False)

            deg = cell.degradation_df
            efc_index = deg.index.to_numpy(float)
            soh_values = deg.values if isinstance(deg, pd.Series) else deg.iloc[:, 0].to_numpy(float)

            efc_cum = hourly_df_all["efc_cycle"].to_numpy(float)
            hourly_df_all["SOH"] = np.interp(efc_cum, efc_index, soh_values)

        hourly_df_all.to_csv(f"./Prediction_Accuracy_Result/Simulation Result/{model_name}-{market_friction}.csv", index=False)
        calib_log_df.to_csv(f"./Prediction_Accuracy_Result/Simulation Log/{model_name}-{market_friction}.csv", index=False)
        print(f"Saved {model_name}-{market_friction}.")
        print(f'Performance: {1 - (hourly_df_all["grid_cost"].sum() / total_baseline_cost):.2%}')

        saving_percentage = 1 - (result_df['grid_cost'].sum() / total_baseline_cost)

        dt_h = 1.0  # 你的仿真是 1 小时间隔

        efc = result_df.iloc[len(result_df)-1]['efc_cycle']

        result_summary_df.loc[len(result_summary_df)] = [f'{model_name}-{market_friction}', market_friction, saving_percentage, efc]

Saved MF-1 Model
Performance: 10.16%
Saved TP-0.95.
Performance: 9.60%
Saved PP-0.95.
Performance: 9.55%


d:\Applications\miniconda3\envs\rl_env2\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Saved PDU-0.95.
Performance: 9.58%
Saved TP-0.9.
Performance: 9.05%
Saved PP-0.9.
Performance: 9.01%
Saved PDU-0.9.
Performance: 9.04%
Saved TP-0.85.
Performance: 8.53%
Saved PP-0.85.
Performance: 8.59%
Saved PDU-0.85.
Performance: 8.55%
Saved TP-0.8.
Performance: 8.03%
Saved PP-0.8.
Performance: 8.14%
Saved PDU-0.8.
Performance: 8.00%
Saved TP-0.75.
Performance: 7.64%
Saved PP-0.75.
Performance: 7.79%
Saved PDU-0.75.
Performance: 7.53%
Saved TP-0.7.
Performance: 7.26%
Saved PP-0.7.
Performance: 7.44%
Saved PDU-0.7.
Performance: 7.10%
Saved TP-0.65.
Performance: 6.92%
Saved PP-0.65.
Performance: 7.19%
Saved PDU-0.65.
Performance: 6.73%
Saved TP-0.6.
Performance: 6.62%
Saved PP-0.6.
Performance: 6.97%
Saved PDU-0.6.
Performance: 6.40%
Saved TP-0.55.
Performance: 6.34%
Saved PP-0.55.
Performance: 6.74%
Saved PDU-0.55.
Performance: 6.06%
Saved TP-0.5.
Performance: 6.07%
Saved PP-0.5.
Performance: 6.57%
Saved PDU-0.5.
Performance: 5.74%


In [10]:
prediction_accuracy_result_summary_df = result_summary_df.copy()
prediction_accuracy_result_summary_df

,Model Name,Market Friction,Saving Percentage,EFC
0,MF-1,1.00,0.101585,298.256207
1,TP-0.95,0.95,0.095982,297.836069
2,PP-0.95,0.95,0.095491,297.829502
3,PDU-0.95,0.95,0.095838,298.057148
4,TP-0.9,0.90,0.090483,298.175919
5,PP-0.9,0.90,0.090116,297.782336
6,PDU-0.9,0.90,0.090358,298.267766
7,TP-0.85,0.85,0.085266,298.441796
8,PP-0.85,0.85,0.085948,297.859560
9,PDU-0.85,0.85,0.085465,298.748563


In [11]:
def cvrmse(y_true, y_pred):
    """
    CV(RMSE) = RMSE / mean(y_measured), per ASHRAE-style normalization.
    If mean(measured) is ~0, result can explode — check mean_actual in output.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[m], y_pred[m]
    if len(y_true) == 0:
        return np.nan, np.nan, 0
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mean_actual = float(np.mean(y_true))
    cv = rmse / mean_actual if mean_actual != 0 else np.nan
    return rmse, cv, len(y_true)
def cvrmse_report(ready_to_test_dict):
    """同一批 datetime 上对齐三个 model，分别算 Demand / PV 的 CVRMSE。"""
    dfs = {k: v.copy() for k, v in ready_to_test_dict.items()}
    for name, df in dfs.items():
        if "datetime_utc" not in df.columns:
            raise ValueError(f"{name}: need datetime_utc")
        df["_dt"] = pd.to_datetime(df["datetime_utc"])
    common = None
    for name, df in dfs.items():
        s = set(df["_dt"])
        common = s if common is None else (common & s)
    common = sorted(common)
    print(f"aligned timestamps: {len(common)}")
    rows = []
    for model_name, df in dfs.items():
        sub = df[df["_dt"].isin(common)].sort_values("_dt")
        for target, pred in [("Demand", "Demand_Predicted"), ("PV", "PV_Predicted")]:
            rmse, cv, n = cvrmse(sub[target].values, sub[pred].values)
            mean_a = float(np.nanmean(sub[target].values))
            rows.append({
                "model": model_name,
                "variable": target,
                "n": n,
                "mean_actual": mean_a,
                "RMSE": rmse,
                "CVRMSE": cv,
                "CVRMSE_pct": cv * 100 if np.isfinite(cv) else np.nan,
            })
    return pd.DataFrame(rows)
# 在定义好 ready_to_test_dict 之后运行：
report_df = cvrmse_report(ready_to_test_dict)
print(report_df.to_string(index=False))

aligned timestamps: 46920
model variable     n  mean_actual      RMSE   CVRMSE  CVRMSE_pct
   TP   Demand 46920   210.211157 45.369697 0.215829   21.582916
   TP       PV 46920    74.200643 42.757717 0.576245   57.624456
   PP   Demand 46920   210.211157  0.000000 0.000000    0.000000
   PP       PV 46920    74.200643  0.000000 0.000000    0.000000
  PDU   Demand 46920   210.211157 84.649583 0.402688   40.268835
  PDU       PV 46920    74.200643 67.054230 0.903688   90.368800


## Sensitivity Analysis: Recalibration Day Gap and Past Weeks Horizon

In [12]:
past_weeks_list = [0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 6, 8, 10, 12, 16]

result_summary_df = pd.DataFrame(columns=['Model', 'Market Friction', 'Past Weeks', 'Saving Percentage', 'EFC'])

for past_weeks in past_weeks_list:
    for market_friction in [1]:
        df_raw = pred_full.copy()
        df_raw["Buy_Price"] = df_raw["Price"]
        df_raw["Sell_Price"] = market_friction * df_raw["Price"]

        # Test run.
        result_df, calib_log_df = run_rolling_5y(
            df_raw=df_raw,
            cache_dir="./battery_model_params_20260214/1_hour_0.0_1.0",
            Ns=96, Np=10, N_series=105, N_parallel=2,
            start="2020-12-27",
            end="2025-12-26",
            past_weeks=past_weeks,              
            soc_grid_size=101,
            action_grid_size=81,
            allow_export=True,
            terminal_equal_init=False,
            n_jobs=-1,
            max_iter_calib=10,
            max_iter_day=10,
            calib_every_days=1,
            use_calibrated_lambda_only=True,   # ← 新增：禁用每日 λ 求解，只用校准 λ，
            return_calibration_log=True,
        )

        hourly_df_all = result_df.copy()
        # ----- 1) baseline_cost_cum：无电池时的累积电费 -----
        dt_h = 1.0
        if "baseline_cost_cum" not in hourly_df_all.columns:
            net_load = hourly_df_all["Demand"].to_numpy(float) - hourly_df_all["PV"].to_numpy(float)
            baseline_cost_hourly = hourly_df_all["Buy_Price"].to_numpy(float) * net_load * dt_h
            hourly_df_all["baseline_cost_cum"] = np.cumsum(baseline_cost_hourly)

        hourly_df_all['grid_cost_cum'] = np.cumsum(hourly_df_all['grid_cost'])

        # ----- 2) SOH：由 cell 的 EFC–SOH 关系根据 EFC_cum_cell 插值 -----
        if "SOH" not in hourly_df_all.columns:
            try:
                cell = system.pack.cell
            except NameError:
                cache_dir = "./battery_model_params_20260214/1_hour_0.0_1.0"
                cell = create_cell_from_stored_json(cache_dir, show_info=False)

            deg = cell.degradation_df
            efc_index = deg.index.to_numpy(float)
            soh_values = deg.values if isinstance(deg, pd.Series) else deg.iloc[:, 0].to_numpy(float)

            efc_cum = hourly_df_all["efc_cycle"].to_numpy(float)
            hourly_df_all["SOH"] = np.interp(efc_cum, efc_index, soh_values)

        hourly_df_all.to_csv(f"./Past_Weeks_Result/Simulation Result/Past Weeks-{past_weeks}.csv", index=False)
        calib_log_df.to_csv(f"./Past_Weeks_Result/Simulation Log/Past Weeks-{past_weeks}.csv", index=False)

        saving_percentage = 1 - (result_df['grid_cost'].sum() / total_baseline_cost)

        dt_h = 1.0  # 你的仿真是 1 小时间隔

        efc = result_df.iloc[len(result_df)-1]['efc_cycle']

        result_summary_df.loc[len(result_summary_df)] = [f'Past Weeks-{past_weeks}', market_friction, past_weeks, saving_percentage, efc]

        print(f'Past Weeks-{past_weeks} done. Improvement: {saving_percentage:.2%}, EFC: {efc:.2f}')
        

Past Weeks-0.25 done. Improvement: 8.12%, EFC: 297.31
Past Weeks-0.5 done. Improvement: 9.59%, EFC: 297.62
Past Weeks-0.75 done. Improvement: 9.39%, EFC: 298.84
Past Weeks-1 done. Improvement: 9.46%, EFC: 298.41
Past Weeks-1.5 done. Improvement: 9.82%, EFC: 298.03
Past Weeks-2 done. Improvement: 10.08%, EFC: 298.42
Past Weeks-3 done. Improvement: 10.16%, EFC: 298.37
Past Weeks-4 done. Improvement: 10.16%, EFC: 298.26
Past Weeks-6 done. Improvement: 10.20%, EFC: 298.14
Past Weeks-8 done. Improvement: 10.17%, EFC: 297.76
Past Weeks-10 done. Improvement: 10.10%, EFC: 297.56
Past Weeks-12 done. Improvement: 10.17%, EFC: 297.92
Past Weeks-16 done. Improvement: 10.09%, EFC: 297.74


In [13]:
past_weeks_result_summary_df = result_summary_df.copy()
past_weeks_result_summary_df

,Model,Market Friction,Past Weeks,Saving Percentage,EFC
0,Past Weeks-0.25,1,0.25,0.081227,297.308022
1,Past Weeks-0.5,1,0.50,0.095888,297.617820
2,Past Weeks-0.75,1,0.75,0.093950,298.836417
3,Past Weeks-1,1,1.00,0.094557,298.409409
4,Past Weeks-1.5,1,1.50,0.098220,298.028373
5,Past Weeks-2,1,2.00,0.100804,298.421832
6,Past Weeks-3,1,3.00,0.101585,298.365565
7,Past Weeks-4,1,4.00,0.101585,298.256207
8,Past Weeks-6,1,6.00,0.102026,298.140065
9,Past Weeks-8,1,8.00,0.101718,297.760773


In [14]:
# Sensitivity Analysis: Recalibration Day Gap
calib_every_days_list = [1, 3, 7, 14, 28, 56, 84, 112, 140, 168, 196, 224, 252, 280, 308, 336, 365]

result_summary_df = pd.DataFrame(columns=['Model', 'Market Friction', 'Recalibration Day Gap', 'Saving Percentage', 'EFC'])

for calib_every_days in calib_every_days_list:
    for market_friction in [1]:
        df_raw = pred_full.copy()
        df_raw["Buy_Price"] = df_raw["Price"]
        df_raw["Sell_Price"] = market_friction * df_raw["Price"]

        # Test run.
        result_df, calib_log_df = run_rolling_5y(
            df_raw=df_raw,
            cache_dir="./battery_model_params_20260214/1_hour_0.0_1.0",
            Ns=96, Np=10, N_series=105, N_parallel=2,
            start="2020-12-27",
            end="2025-12-26",
            past_weeks=4,              
            soc_grid_size=101,
            action_grid_size=81,
            allow_export=True,
            terminal_equal_init=False,
            n_jobs=-1,
            max_iter_calib=10,
            max_iter_day=10,
            calib_every_days=calib_every_days,
            use_calibrated_lambda_only=True,   # ← 新增：禁用每日 λ 求解，只用校准 λ，
            return_calibration_log=True,
        )

        hourly_df_all = result_df.copy()
        # ----- 1) baseline_cost_cum：无电池时的累积电费 -----
        dt_h = 1.0
        if "baseline_cost_cum" not in hourly_df_all.columns:
            net_load = hourly_df_all["Demand"].to_numpy(float) - hourly_df_all["PV"].to_numpy(float)
            baseline_cost_hourly = hourly_df_all["Buy_Price"].to_numpy(float) * net_load * dt_h
            hourly_df_all["baseline_cost_cum"] = np.cumsum(baseline_cost_hourly)

        hourly_df_all['grid_cost_cum'] = np.cumsum(hourly_df_all['grid_cost'])

        # ----- 2) SOH：由 cell 的 EFC–SOH 关系根据 EFC_cum_cell 插值 -----
        if "SOH" not in hourly_df_all.columns:
            try:
                cell = system.pack.cell
            except NameError:
                cache_dir = "./battery_model_params_20260214/1_hour_0.0_1.0"
                cell = create_cell_from_stored_json(cache_dir, show_info=False)

            deg = cell.degradation_df
            efc_index = deg.index.to_numpy(float)
            soh_values = deg.values if isinstance(deg, pd.Series) else deg.iloc[:, 0].to_numpy(float)

            efc_cum = hourly_df_all["efc_cycle"].to_numpy(float)
            hourly_df_all["SOH"] = np.interp(efc_cum, efc_index, soh_values)

        hourly_df_all.to_csv(f"./Recalibration_Day_Gap_Result/Simulation Result/Recalibration Day Gap-{calib_every_days}.csv", index=False)
        calib_log_df.to_csv(f"./Recalibration_Day_Gap_Result/Simulation Log/Recalibration Day Gap-{calib_every_days}.csv", index=False)

        saving_percentage = 1 - (result_df['grid_cost'].sum() / total_baseline_cost)

        dt_h = 1.0  # 你的仿真是 1 小时间隔


        efc = result_df.iloc[len(result_df)-1]['efc_cycle']

        result_summary_df.loc[len(result_summary_df)] = [f'Recalibration Day Gap-{calib_every_days}', market_friction, calib_every_days, saving_percentage, efc]

        print(f'Recalibration Day Gap-{calib_every_days} done. Improvement: {saving_percentage:.2%}, EFC: {efc:.2f}')


Recalibration Day Gap-1 done. Improvement: 10.16%, EFC: 298.26
Recalibration Day Gap-3 done. Improvement: 10.15%, EFC: 297.07
Recalibration Day Gap-7 done. Improvement: 10.09%, EFC: 296.60
Recalibration Day Gap-14 done. Improvement: 10.03%, EFC: 294.65
Recalibration Day Gap-28 done. Improvement: 9.91%, EFC: 293.02
Recalibration Day Gap-56 done. Improvement: 9.66%, EFC: 297.97
Recalibration Day Gap-84 done. Improvement: 9.41%, EFC: 300.55
Recalibration Day Gap-112 done. Improvement: 9.73%, EFC: 296.89
Recalibration Day Gap-140 done. Improvement: 11.15%, EFC: 302.47
Recalibration Day Gap-168 done. Improvement: 11.31%, EFC: 301.93
Recalibration Day Gap-196 done. Improvement: 11.32%, EFC: 351.95
Recalibration Day Gap-224 done. Improvement: 12.02%, EFC: 372.13
Recalibration Day Gap-252 done. Improvement: 11.89%, EFC: 383.56
Recalibration Day Gap-280 done. Improvement: 11.79%, EFC: 341.60
Recalibration Day Gap-308 done. Improvement: 11.65%, EFC: 352.85
Recalibration Day Gap-336 done. Improve

In [15]:
recalibration_day_gap_result_summary_df = result_summary_df.copy()
recalibration_day_gap_result_summary_df

,Model,Market Friction,Recalibration Day Gap,Saving Percentage,EFC
0,Recalibration Day Gap-1,1,1,0.101585,298.256207
1,Recalibration Day Gap-3,1,3,0.101501,297.069612
2,Recalibration Day Gap-7,1,7,0.100865,296.598560
3,Recalibration Day Gap-14,1,14,0.100339,294.645351
4,Recalibration Day Gap-28,1,28,0.099055,293.015066
5,Recalibration Day Gap-56,1,56,0.096589,297.971945
6,Recalibration Day Gap-84,1,84,0.094131,300.553232
7,Recalibration Day Gap-112,1,112,0.097276,296.891540
8,Recalibration Day Gap-140,1,140,0.111522,302.469989
9,Recalibration Day Gap-168,1,168,0.113063,301.934407


## END 